Step 0 — Set up the catalog, schema and table name


In [0]:
CATALOG = "workspace"
SCHEMA = "module7_delta_lake"
TABLE = "orders"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Target table:", FULL_TABLE_NAME)



Step 1 — Import data types and create the sample orders

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType, DateType
from datetime import date
orders_schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("customer_id", LongType(), False),
    StructField("customer_name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("order_date", DateType(), True),
])
orders_data = [
    (1001, 501, "Aarav", "Laptop", "Electronics", 72000.0, "COMPLETED", date(2026, 10, 1)),
    (1002, 502, "Diya", "Office Chair", "Furniture", 8500.0, "COMPLETED", date(2026, 10, 2)),
    (1003, 503, "Kabir", "Headphones", "Electronics", 4500.0, "PENDING", date(2026, 10, 3)),
    (1004, 501, "Aarav", "Desk", "Furniture", 12500.0, "COMPLETED", date(2026, 10, 4)),
    (1005, 504, "Meera", "Smart Watch", "Electronics", 9800.0, "CANCELLED", date(2026, 10, 5)),
    (1006, 505, "Ishaan", "Bookshelf", "Furniture", 6200.0, "PENDING", date(2026, 10, 6)),
    (1007, 502, "Diya", "Keyboard", "Electronics", 2200.0, "COMPLETED", date(2026, 10, 7)),
    (1008, 506, "Anaya", "Dining Table", "Furniture", 24000.0, "COMPLETED", date(2026, 10, 8)),
]
orders_df = spark.createDataFrame(orders_data, schema=orders_schema)
display(orders_df)



Step 2 — Save the DataFrame as a Delta table

In [0]:
(orders_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME))
display(spark.table(FULL_TABLE_NAME))

Step 3 — View the Delta table history

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

Step 4 — Insert, update and delete records

In [0]:
spark.sql(f"""
INSERT INTO {FULL_TABLE_NAME}
(order_id, customer_id, customer_name, product, category, amount, status, order_date)
VALUES
(1009, 507, 'Rohan', 'Monitor', 'Electronics', 18500.0, 'COMPLETED', DATE '2026-10-09')
""")
spark.sql(f"UPDATE {FULL_TABLE_NAME} SET status = 'COMPLETED' WHERE order_id = 1003")
spark.sql(f"DELETE FROM {FULL_TABLE_NAME} WHERE status = 'CANCELLED'")
display(spark.table(FULL_TABLE_NAME).orderBy("order_id"))



Step 5 — Check the transaction history again

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

Step 6 — Build a deliberately incompatible incoming dataset

In [0]:
bad_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", StringType(), True),  # Deliberately incompatible type
    StructField("status", StringType(), True),
    StructField("order_date", StringType(), True),
])
bad_data = [(1010, 508, "Sara", "Tablet", "Electronics", "not-a-number", "PENDING", "2026-10-10")]
bad_df = spark.createDataFrame(bad_data, schema=bad_schema)
try:
    bad_df.write.format("delta").mode("append").saveAsTable(FULL_TABLE_NAME)
    print("This runtime accepted or coerced the incoming schema. Inspect the table schema and row.")
except Exception as exc:
    print("The incompatible append was rejected by the table/runtime.")
    print(str(exc)[:700])



Step 7 — Add a column through schema evolution

In [0]:
evolved_schema = orders_schema.add(StructField("loyalty_tier", StringType(), True))
evolved_data = [
    (1011, 509, "Vivaan", "Phone", "Electronics", 42000.0, "COMPLETED", date(2026, 10, 11), "GOLD"),
    (1012, 510, "Myra", "Sofa", "Furniture", 55000.0, "PENDING", date(2026, 10, 12), "SILVER"),
]
evolved_df = spark.createDataFrame(evolved_data, schema=evolved_schema)
(evolved_df.write.format("delta").mode("append")
    .option("mergeSchema", "true").saveAsTable(FULL_TABLE_NAME))
spark.table(FULL_TABLE_NAME).printSchema()
display(spark.table(FULL_TABLE_NAME).orderBy("order_id"))

Step 8 — Review available versions

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

Step 9 — Read the original version using Time Travel


In [0]:
try:
    display(spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} VERSION AS OF 0").orderBy("order_id"))
except Exception as exc:
    print("Version 0 is unavailable. Select a valid version from DESCRIBE HISTORY.")
    print(str(exc)[:700])

Step 10 — Compare historical and current row counts

In [0]:
try:
    old_count = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_TABLE_NAME} VERSION AS OF 0").first()["n"]
    current_count = spark.table(FULL_TABLE_NAME).count()
    print("Version 0 rows:", old_count)
    print("Current rows:", current_count)
except Exception as exc:
    print("Choose an available version from DESCRIBE HISTORY.")
    print(str(exc)[:500])



Step 11 — Run OPTIMIZE

In [0]:
try:
    spark.sql(f"OPTIMIZE {FULL_TABLE_NAME}")
    print("OPTIMIZE completed.")
except Exception as exc:
    print("OPTIMIZE unavailable or rejected in this workspace/runtime:")
    print(str(exc)[:700])

Step 12 — Attempt Z-Order optimization

In [0]:
try:
    spark.sql(f"OPTIMIZE {FULL_TABLE_NAME} ZORDER BY (customer_id)")
    print("Z-Order completed.")
except Exception as exc:
    print("Z-Order unavailable or rejected in this workspace/runtime:")
    print(str(exc)[:700])

Step 13 — Validate the final table

In [0]:
spark.table(FULL_TABLE_NAME).printSchema()
print("Final row count:", spark.table(FULL_TABLE_NAME).count())
display(spark.table(FULL_TABLE_NAME).orderBy("order_id"))

Displays the resulting DataFrame or query output in the Databricks notebook.


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

